In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42


: 

In [ ]:
path = "synthetic_cardiac_dataset_2.csv"   
df = pd.read_csv(path)

df.shape, df.head(5)


In [ ]:
print("shape:", df.shape)

# Target
print("\nTarget value counts:")
print(df["Target"].value_counts(dropna=False))

# dtypes
print("\nDtypes:")
print(df.dtypes)

# missing
miss = df.isna().mean().sort_values(ascending=False)
print("\nMissing % (top 30):")
print((miss.head(30)*100).round(2))

# numeric ranges
num_cols = df.select_dtypes(include=[np.number]).columns
print("\nNumeric min/max:")
print(df[num_cols].agg(["min","max"]).T)

In [ ]:
# Validation rules based on domain logic (initial – will be refined)
validation_rules = {
    "Age": {"min": 0, "max": 120},
    "Height": {"min": 50, "max": 250},      # cm
    "Weight": {"min": 20, "max": 300},      # kg
    "BMI": {"min": 10, "max": 80},
    "RestingBP": {"min": 50, "max": 300},
    "Cholesterol": {"min": 50, "max": 600},
    "MaxHR": {"min": 40, "max": 250},
    "CAD": {"allowed": [0,1,2,3,4]},
    "Target": {"allowed": [0,1]}
}


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns

df[num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T


In [ ]:
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for c in cat_cols:
    print(f"\n--- {c} ---")
    display(df[c].value_counts(dropna=False))


In [ ]:
def normalize_binary(series):
    s = series.copy()

    # اگر متنی است
    if s.dtype == "O":
        s = s.astype(str).str.strip().str.lower()
        s = s.replace({
            "1": 1, "true": 1, "yes": 1, "y": 1, "بله": 1,
            "0": 0, "false": 0, "no": 0, "n": 0, "خیر": 0
        })

    return s


In [ ]:
binary_cols = ["Sex", "Heart_Failure", "Diabetes", "Smoking", "CHD"]

for c in binary_cols:
    if c in df.columns:
        df[c] = normalize_binary(df[c])

for c in binary_cols:
    if c in df.columns:
        print(f"\n{c}")
        display(df[c].value_counts(dropna=False))


In [ ]:
df["Arrhythmia"] = (
    df["Arrhythmia"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "PVC": "PVC",
        "AF": "AF",
        "SVT": "SVT",
        "NONE": "NONE",
        "NAN": np.nan
    })
)

df["Arrhythmia"].value_counts(dropna=False)


In [ ]:
df["Valvular_Disease"] = (
    df["Valvular_Disease"]
    .astype(str)
    .str.strip()
    .replace({
        "خفبف": "خفیف",
        "وسط": "متوسط",
        "حاد": "شدید"
    })
)

df["Valvular_Disease"].value_counts(dropna=False)


In [ ]:
df["Education_Level"] = df["Education_Level"].fillna("Unknown")

df["Education_Level"].value_counts()


In [ ]:
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"], errors="coerce")
df["Last_Visit_Date"] = pd.to_datetime(df["Last_Visit_Date"], errors="coerce")

df[["Admission_Date", "Last_Visit_Date"]].info()


In [ ]:
df["Followup_Days"] = (df["Last_Visit_Date"] - df["Admission_Date"]).dt.days

df["Followup_Days"].describe()
df[df["Followup_Days"] < 0].head()


In [ ]:
issue_log = []

# Age خارج از بازه
age_out = (df["Age"] > 80) | (df["Age"] < 15)
issue_log.append({
    "feature": "Age",
    "issue": "out_of_range",
    "count": int(age_out.sum())
})

# MaxHR خارج از بازه
maxhr_out = df["MaxHR"] > 200
issue_log.append({
    "feature": "MaxHR",
    "issue": "out_of_range",
    "count": int(maxhr_out.sum())
})

# Alcoholism مقدار 99
alc_out = df["Alcoholism"] > 50
issue_log.append({
    "feature": "Alcoholism",
    "issue": "invalid_value_>50",
    "count": int(alc_out.sum())
})

pd.DataFrame(issue_log)


In [ ]:
for c in ["Sex", "Heart_Failure", "Diabetes", "Smoking", "CHD"]:
    print(c)
    display(df[c].value_counts(dropna=False))


In [ ]:
# اصلاح Sex
df["Sex"] = (
    df["Sex"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"m": 1, "f": 0})
)

# اصلاح CHD
df["CHD"] = df["CHD"].replace({True: 1, False: 0})


In [ ]:
for c in ["Sex", "CHD"]:
    print(c)
    display(df[c].value_counts(dropna=False))


In [ ]:
df["Arrhythmia"].value_counts(dropna=False)


In [ ]:
df["Valvular_Disease"].value_counts(dropna=False)


In [ ]:
df["Education_Level"].value_counts(dropna=False)


In [ ]:
df[["Admission_Date", "Last_Visit_Date"]].isna().sum()



In [ ]:
df["Followup_Days"].describe()


In [ ]:
df[df["Followup_Days"] < 0].shape


In [ ]:
pd.DataFrame(issue_log)


In [ ]:
# Age
df.loc[df["Age"] > 80, "Age"] = 80

# MaxHR
df.loc[df["MaxHR"] > 200, "MaxHR"] = 200

# Alcoholism
df.loc[df["Alcoholism"] > 50, "Alcoholism"] = 0


In [ ]:
df["Arrhythmia"] = df["Arrhythmia"].fillna("NONE")
df["Arrhythmia"].value_counts()


In [ ]:
df["Diabetes"] = df["Diabetes"].fillna(0)
df["Diabetes"].value_counts()


In [ ]:
df["Smoking"] = df["Smoking"].fillna(0)
df["Smoking"].value_counts()


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:

df["Last_Visit_Date"] = df["Last_Visit_Date"].fillna(df["Admission_Date"])


df = df.drop(columns=["Admission_Date", "Followup_Days"])


In [ ]:
df.isna().sum()


In [ ]:
df = df.drop(columns=["Last_Visit_Date"])
df.isna().sum()


In [ ]:
TARGET_COL = "Target"

y = df[TARGET_COL].copy()
X = df.drop(columns=["ID", TARGET_COL])


In [ ]:
education_map = {
    "Illiterate": 0,
    "Primary": 1,
    "Secondary": 2,
    "High School": 3,
    "Bachelor+": 4,
    "Unknown": 2  
}

valvular_map = {
    "بدون": 0,
    "خفیف": 1,
    "متوسط": 2,
    "شدید": 3
}


In [ ]:
X["Education_Level"] = X["Education_Level"].map(education_map)
X["Valvular_Disease"] = X["Valvular_Disease"].map(valvular_map)

X[["Education_Level", "Valvular_Disease"]].describe()


In [ ]:
nominal_cols = ["Hospital", "CPT", "Arrhythmia", "Department"]

X = pd.get_dummies(X, columns=nominal_cols, drop_first=False)


In [ ]:
X.shape, y.shape


In [ ]:
X.dtypes.value_counts()


In [ ]:
X.select_dtypes(include=["object"]).columns


In [ ]:
for c in ["Sex", "Diabetes", "Smoking"]:
    X[c] = pd.to_numeric(X[c], errors="coerce").astype("Int64")  

X[["Sex", "Diabetes", "Smoking"]] = X[["Sex", "Diabetes", "Smoking"]].fillna(0).astype(int)


In [ ]:
X.dtypes.value_counts(), X[["Sex","Diabetes","Smoking"]].head()


In [ ]:
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X.dtypes.value_counts()


In [ ]:
final_df = X.copy()
final_df["Target"] = y.values
final_df.to_csv("final_clean.csv", index=False)
final_df.shape
